In [5]:
pip install fugashi ipadic

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install unidic-lite

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 17.5 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658912 sha256=285761e70602dd3710b0bc8733233d20cb1409a0511b6059e01be26d8a4c8a6b
  Stored in directory: /home/qqqlq-desk/.cache/pip/wheels/5e/1f/0f/4d43887e5476d956fae828ee9b6687becd5544d68b51ed633d
Successfully built unidic-lite
Note: you may need to restart the kernel to use updated packages.


In [8]:
from transformers import pipeline

# 東北大学の日本語BERTで感情分析
classifier = pipeline(
    "sentiment-analysis",
    model="koheiduck/bert-japanese-finetuned-sentiment"
)

# 試してみる
texts = [
    "この映画はとても面白かった！",
    "最悪な映画だった。時間の無駄。",
    "まあまあかな。普通だった。",
    "感動して泣いてしまいました。素晴らしい作品です。",
]

results = classifier(texts)
for text, result in zip(texts, results):
    print(f"{result['label']} ({result['score']:.3f}): {text}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: koheiduck/bert-japanese-finetuned-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


POSITIVE (0.993): この映画はとても面白かった！
NEGATIVE (0.995): 最悪な映画だった。時間の無駄。
NEUTRAL (0.944): まあまあかな。普通だった。
POSITIVE (0.970): 感動して泣いてしまいました。素晴らしい作品です。


In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "koheiduck/bert-japanese-finetuned-sentiment"
)

texts = [
    "この映画はとても面白かった！",
    "感動して泣いてしまいました。",
    "この映画はすごく面白い。"
]

for text in texts:
    tokens = tokenizer.tokenize(text)
    print(text)
    print(f"→ {tokens}")
    print()

この映画はとても面白かった！
→ ['この', '映画', 'は', 'とても', '面白', '##かっ', 'た', '!']

感動して泣いてしまいました。
→ ['感動', 'し', 'て', '泣い', 'て', 'しまい', 'まし', 'た', '。']

この映画はすごく面白い。
→ ['この', '映画', 'は', 'すご', '##く', '面白い', '。']



In [12]:
words = ["すごい", "すごく", "面白い", "面白かった", "面白く"]
for word in words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:10} → {tokens}")

すごい        → ['すごい']
すごく        → ['すご', '##く']
面白い        → ['面白い']
面白かった      → ['面白', '##かっ', 'た']
面白く        → ['面白', '##く']


In [13]:
# 映画レビューを分類して集計する
reviews = [
    "最高の映画でした！また見たいです。",
    "退屈すぎて途中で寝てしまった。",
    "普通かな。特に印象に残らなかった。",
    "主演俳優の演技が素晴らしかった！",
    "ストーリーが意味不明で全然楽しめなかった。",
    "家族で楽しめる良い映画でした。",
    "期待外れだった。もう少し頑張ってほしい。",
    "感動的なシーンが多くて泣けた。",
]

results = classifier(reviews)

# 集計する
from collections import Counter
labels = [r['label'] for r in results]
counts = Counter(labels)

print("分類結果:")
for review, result in zip(reviews, results):
    print(f"  {result['label']:8} ({result['score']:.2f}): {review}")

print(f"\n集計: {dict(counts)}")

分類結果:
  POSITIVE (0.99): 最高の映画でした！また見たいです。
  NEGATIVE (0.94): 退屈すぎて途中で寝てしまった。
  NEGATIVE (0.78): 普通かな。特に印象に残らなかった。
  POSITIVE (0.99): 主演俳優の演技が素晴らしかった！
  NEGATIVE (0.99): ストーリーが意味不明で全然楽しめなかった。
  POSITIVE (0.99): 家族で楽しめる良い映画でした。
  NEGATIVE (0.72): 期待外れだった。もう少し頑張ってほしい。
  POSITIVE (0.93): 感動的なシーンが多くて泣けた。

集計: {'POSITIVE': 4, 'NEGATIVE': 4}


In [23]:
# 単独で試す
text = "普通かな。特に印象に残らなかった。"
result_single = classifier(text)
print(text)
print("単独:", result_single)
text = "まあまあかな。普通だった。"
result_single = classifier(text)
print(text)
print("単独:", result_single)
text = "アクションは良かったけど、内容はまあまかな"
result_single = classifier(text)
print(text)
print("単独:", result_single)


# バッチで試す
result_batch = classifier([
    "普通かな。特に印象に残らなかった。",
    "まあまあかな。普通だった。",
    "アクションは良かったけど、内容はまあまかな",
])
print("バッチ:", result_batch)

普通かな。特に印象に残らなかった。
単独: [{'label': 'NEGATIVE', 'score': 0.7765659093856812}]
まあまあかな。普通だった。
単独: [{'label': 'NEUTRAL', 'score': 0.9443663358688354}]
アクションは良かったけど、内容はまあまかな
単独: [{'label': 'NEGATIVE', 'score': 0.8928399682044983}]
バッチ: [{'label': 'NEGATIVE', 'score': 0.7765659093856812}, {'label': 'NEUTRAL', 'score': 0.9443663358688354}, {'label': 'NEGATIVE', 'score': 0.8928399682044983}]
